In [1]:
from numba import cuda
import numpy as np

In [2]:
@cuda.jit
def add_arrays(a, b, c):
    i = cuda.grid(1)      # <--- Unique ID for every thread
    if i < a.size:        # <--- Safety check
        c[i] = a[i] + b[i] # <--- The actual work

In [3]:
n = 1024

# Create input arrays
a = np.arange(n, dtype=np.float32)
b = np.arange(n, dtype=np.float32) # Fixed the PDF typo here
c = np.zeros(n, dtype=np.float32)

# Allocate memory on GPU
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.device_array_like(c)

# Calculate blocks needed
threads_per_block = 256
blocks_per_grid = (n + (threads_per_block - 1)) // threads_per_block

# Launch the kernel
add_arrays[blocks_per_grid, threads_per_block](d_a, d_b, d_c)

# Copy result back to CPU
c = d_c.copy_to_host()

# 3. Print exactly as requested
# The PDF uses default numpy printing for the first 10 elements
print("First 10 results:", c[:10])

c:\Users\dil\AppData\Local\Programs\Python\Python313\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:690: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


First 10 results: [ 0.  2.  4.  6.  8. 10. 12. 14. 16. 18.]


In [4]:
@cuda.jit
def add_matrix_2d(A, B, C):

    row, col = cuda.grid(2)

    
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = A[row, col] + B[row, col]

rows = 4
cols = 4


a = np.ones((rows, cols), dtype=np.float32)
b = np.full((rows, cols), 2.0, dtype=np.float32)
c = np.zeros((rows, cols), dtype=np.float32)

d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.device_array_like(c)


threads_per_block = (16, 16)


blocks_per_grid_x = (rows + threads_per_block[0] - 1) // threads_per_block[0]
blocks_per_grid_y = (cols + threads_per_block[1] - 1) // threads_per_block[1]
blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)


add_matrix_2d[blocks_per_grid, threads_per_block](d_a, d_b, d_c)


c = d_c.copy_to_host()

print("Matrix A (Ones):")
print(a)
print("\nMatrix B (Twos):")
print(b)
print("\nResult Matrix C (A + B):")
print(c)

Matrix A (Ones):
[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]]

Matrix B (Twos):
[[2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]]

Result Matrix C (A + B):
[[3. 3. 3. 3.]
 [3. 3. 3. 3.]
 [3. 3. 3. 3.]
 [3. 3. 3. 3.]]


c:\Users\dil\AppData\Local\Programs\Python\Python313\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:690: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
